In [1]:

# Imports
import os
import pandas as pd
import NWB_reader_functions as nwb_reader
import neural_utils as nutils



ROOT_PATH_AXEL = os.path.join(r'\\sv-nas1.rcp.epfl.ch', 'Petersen-Lab', 'analysis', 'Axel_Bisi', 'NWBFull_bis')
ROOT_PATH_MYRIAM = os.path.join(r'\\sv-nas1.rcp.epfl.ch', 'Petersen-Lab', 'analysis', 'Myriam_Hamon',
                                'NWB')

In [2]:
from glm_utils import *

In [ ]:
single_mouse = True
multiple_mice = False
joint_analysis = True
expert_day = False

# Set paths
experimenter = 'Myriam_Hamon'

proc_data_path = os.path.join('\\\\sv-nas1.rcp.epfl.ch', 'Petersen-Lab', 'analysis', experimenter, 'data', 'processed_data')
if experimenter == 'Axel_Bisi':
     all_nwb_names = os.listdir(ROOT_PATH_AXEL)
elif experimenter == 'Myriam_Hamon':
    all_nwb_names_bis = os.listdir(ROOT_PATH_MYRIAM)
# all_nwb_names.extend(all_nwb_names_bis)
all_nwb_names = os.listdir(ROOT_PATH_AXEL)
all_nwb_mice = [name.split('_')[0] for name in all_nwb_names]

if joint_analysis:
    info_path = os.path.join(r'\\sv-nas1.rcp.epfl.ch', 'Petersen-Lab', 'z_LSENS', 'Share', f'Axel_Bisi_Share',
                             'dataset_info')
    output_path = os.path.join(r'\\sv-nas1.rcp.epfl.ch', 'Petersen-Lab', 'analysis', experimenter,
                               'combined_results')
    # if experimenter == 'Axel_Bisi':
    #     nwb_names_bis = os.listdir(ROOT_PATH_MYRIAM)
    # elif experimenter == 'Myriam_Hamon':
    #     nwb_names_bis = os.listdir(ROOT_PATH_AXEL)
    # all_nwb_names.extend(nwb_names_bis)
    #all_nwb_mice.extend([name.split('_')[0] for name in myriam_nwb_names])
    mouse_info_path = os.path.join(info_path, 'joint_mouse_reference_weight.xlsx')

else:
    info_path = os.path.join('\\\\sv-nas1.rcp.epfl.ch', 'Petersen-Lab', 'analysis', experimenter, 'mice_info')
    output_path = os.path.join('\\\\sv-nas1.rcp.epfl.ch', 'Petersen-Lab', 'analysis', experimenter, 'results')
    mouse_info_path = os.path.join(info_path, 'mouse_reference_weight.xlsx')


mouse_info_df = pd.read_excel(mouse_info_path)
mouse_info_df.rename(columns={'mouse_name': 'mouse_id'}, inplace=True)
# Filter for usable mice
mouse_info_df = mouse_info_df[
    (mouse_info_df['exclude'] == 0) &
    (mouse_info_df['reward_group'].isin(['R+', 'R-'])) &
    (mouse_info_df['recording'] == 1)

    ]

# Show mouse count per reward group
for group in mouse_info_df['reward_group'].unique():
    count = len(mouse_info_df[mouse_info_df['reward_group'] == group])
    print(f"Reward group {group} has {count} mice.")

# Filter by available NWB files
subject_ids = mouse_info_df['mouse_id'].unique()
subject_ids = [mouse for mouse in subject_ids if any(mouse in name for name in all_nwb_mice)]

# Exclude specific mice
# excluded_mice = ['MH006', 'MH038'] #invalid NWB file 006, 038 ephys_exclude
# subject_ids = [s for s in subject_ids if s not in excluded_mice]

#subject_ids = [f'AB{str(i).zfill(3)}' for i in range(116,158)] # ephys-aligned
# subject_ids = ['AB162', 'AB163', 'AB164']
subject_ids = ['AB086']

print(f"Subject IDs to do: {subject_ids}")

# subject_ids = ['AB131']

### --------------------
# Define analyses to do
### -------------------

# Single-mouse analyses
analyses_to_do_single = ['unit_raster', 'roc_analysis', 'xcorr_analysis']
analyses_to_do_single = ['unit_glm']

# Multi-mouse analyses
analyses_to_do_multi = ['rsu_vs_fsu']
analyses_to_do_multi = ['unit_labels_processing', 'unit_anat_processing']
analyses_to_do_multi = ['unit_anat_processing', 'area_pairs_describe']


# --------------
# Load NWB files
# --------------

nwb_list = [os.path.join(ROOT_PATH_AXEL, name) for name in all_nwb_names if name.startswith('AB')]
nwb_list.extend([os.path.join(ROOT_PATH_MYRIAM, name) for name in all_nwb_names if name.startswith('MH')])
nwb_list = [nwb for nwb in nwb_list if any(subj in nwb for subj in subject_ids)]
trial_table, unit_table, nwb_neural_files = nutils.combine_ephys_nwb(nwb_list, max_workers=24)

# ----------------------------------------
# Perform analyses for each mouse NWB file
# ----------------------------------------

if single_mouse:
    for subject_id in subject_ids:
        print(f"Subject ID : {subject_id}")
        # Create results  folders for the subject
        mouse_results_path = os.path.join(output_path, subject_id)
        os.makedirs(mouse_results_path, exist_ok=True)

        nwb_files = [nwb for nwb in nwb_neural_files if subject_id in nwb]
        if not nwb_files:
            print(f"No NWB files found for {subject_id}")
            continue
        for nwb_file in nwb_files:
            # Create ephys day folder for the session
            beh,day = nwb_reader.get_bhv_type_and_training_day_index(nwb_file)
            mouse_output_path = os.path.join(mouse_results_path, f'{beh}_{day}')
            os.makedirs(mouse_output_path, exist_ok=True)


Reward group R+ has 49 mice.
Reward group R- has 40 mice.
Subject IDs to do: ['AB086']


Loading NWB files:  90%|█████████ | 9/10 [00:02<00:00,  4.40it/s]

In [6]:
import NWB_reader_functions as nwbreader
mouse_id = nwbreader.get_mouse_id(nwb_file)


C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_angle': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_distance': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_likelihood': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_ref_likelihood': Length of data does not match length o

In [8]:
import pandas as pd
import NWB_reader_functions as nwbreader
# -----------------------------------
# Load and prepare data from NWB file
# -----------------------------------

# Get trial table, input and output formatted for GLM, list of predictor types
trials_df = nwbreader.get_trial_table(nwb_file)
trials_df = trials_df[(trials_df['context'] != 'passif') &(trials_df['perf'] != 6)].copy()
trials_df['mouse_id'] = mouse_id
trials_df = load_perf_blocks(trials_df, mouse_id)

trials_df = trials_df.reset_index(drop=True)
#try: #uncomment when design matrix fixed, so that no need to recompute it
#    X, spikes, feature_names = load_model_input_output(output_dir)
#

#  Get trial table, input and output formatted for GLM, list of predictor types
spikes, predictors, predictor_types, n_bins, bin_size, neurons_ccf, scale = load_nwb_spikes_and_predictors(nwb_file, bin_size=BIN_SIZE)
event_defs = predictor_types['event_defs']
analog_keys = predictor_types['analog_keys']

# Build design matrix for entire dataset
X, feature_names = build_design_matrix(predictors, event_defs, analog_keys, bin_size=BIN_SIZE, scale = None)
print(np.isnan(X).mean())
print(np.isnan(X))


C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_angle': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_distance': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_likelihood': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_ref_likelihood': Length of data does not match length o

Keeping active trials and removing auditory onset blocks. Getting whisker trial indices...


C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_angle': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_distance': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_likelihood': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn(error_msg)
C:\Users\mhamon\AppData\Local\anaconda3\envs\ephys_utils_py39\lib\site-packages\pynwb\core.py:52: UserWarning: TimeSeries 'jaw_ref_likelihood': Length of data does not match length o

Keeping active trials and removing auditory onset blocks. Getting whisker trial indices...
CAREFUL THIS IS ONLY 2 RANDOM UNITS
Loading jaw onset data...
[[12.51811815 12.96381738 12.81530637 ... 12.37147588 12.73296804
  12.92385814]
 [20.28960997 19.57715791 17.80523072 ... 17.61224818 17.61719046
  17.7927478 ]
 [16.20970581 16.37159047 15.48706157 ... 16.36793987 15.93898126
  15.34604093]
 ...
 [12.8353066  12.61867902 13.36171259 ... 13.16161938 13.49709707
  13.5137715 ]
 [13.01100482 13.39410261 13.54113877 ... 11.50026205 12.75241977
  13.60807008]
 [        nan         nan         nan ...         nan         nan
          nan]]
[[ 2.44274366e-02  1.78827506e+00 -3.15237915e-01 ...  2.96218886e-01
   6.87160962e-01 -1.42414748e-01]
 [-2.36889786e+01  3.35025655e+01 -7.22490538e+00 ... -1.58922825e-01
   1.24920801e+00  2.17697783e+00]
 [-2.11067291e-01 -4.10201267e-01 -1.25148417e+00 ... -1.07925328e-01
   1.79393782e-01 -2.69991487e-01]
 ...
 [-2.26117114e+00 -1.88192166e+00 -

In [9]:
%matplotlib inline
